# Notebook 4: MLOps Deployment — MLflow & Vertex AI

## AuraCart Retail Analytics — Production Deployment Pipeline

This notebook documents the production deployment workflow for AuraCart's customer segment prediction model:

1. **MLflow Experiment Tracking** — Review and select the champion model
2. **Model Packaging** — Create a unified Scikit-learn Pipeline (preprocessor + classifier)
3. **Artifact Serialization** — Save as `model.joblib` with `requirements.txt`
4. **Google Cloud Storage** — Upload artifacts to GCS bucket
5. **Vertex AI Deployment** — Deploy as a live RESTful prediction endpoint
6. **Endpoint Testing** — Send a prediction request and verify the response

## Step 1: Setup & Load Model Artifacts

In [13]:
from sklearn.preprocessing import FunctionTransformer
import pandas as pd
import numpy as np
import os
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

import mlflow
import mlflow.sklearn

ARTIFACTS_DIR = os.path.join('..', 'artifacts')

print('Libraries loaded successfully.')

Libraries loaded successfully.


## Step 2: Review MLflow Experiments & Select Champion Model

We query the MLflow tracking server to compare all experiment runs and select the best-performing customer segment classification model based on F1-score.

In [14]:
# Connect to MLflow
mlflow.set_tracking_uri('mlruns')

# List all experiments
experiments = mlflow.search_experiments()
print('=== MLflow Experiments ===')
for exp in experiments:
    print(f'  [{exp.experiment_id}] {exp.name}')

=== MLflow Experiments ===
  [983416955902204347] AuraCart_Segment_Classification
  [689952403441876587] AuraCart_Customer_Clustering
  [988440631084945824] AuraCart_Delivery_Classification
  [788470696647428989] AuraCart_Price_Regression
  [0] Default


In [15]:
# Query Customer Segment classification runs
experiment = mlflow.get_experiment_by_name('AuraCart_Segment_Classification')
if experiment:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id],
                               order_by=['metrics.f1_macro DESC'])
    print('=== Customer Segment Classification Runs ===')
    if len(runs) > 0:
        display_cols = ['run_id', 'params.solver', 'params.C', 'params.class_weight',
                        'metrics.accuracy', 'metrics.f1_macro']
        available_cols = [c for c in display_cols if c in runs.columns]
        print(runs[available_cols].to_string())
        
        # Best run
        best_run = runs.iloc[0]
        print(f'\n\u2705 Best Run ID: {best_run["run_id"]}')
        print(f'   F1 Macro: {best_run.get("metrics.f1_macro", "N/A")}')
    else:
        print('No runs found. Please run Notebook 2 first.')
else:
    print('Experiment not found. Please run Notebook 2 first.')

=== Customer Segment Classification Runs ===
                              run_id params.solver params.C params.class_weight  metrics.accuracy  metrics.f1_macro
0   120ab0a62f8d46aa83aee3fb525e6f53         lbfgs      0.1            balanced             0.259          0.236511
1   291a0ad6c5b94eacafa15f4c85ce519c          saga      1.0            balanced             0.259          0.236511
2   a2115743c8e94da2a62b3189c84840eb         lbfgs      0.1            balanced             0.259          0.236511
3   ca9828d931004fcf9eec5106e0786b04          saga      1.0            balanced             0.259          0.236511
4   87dc16aeaa1444a9b577627d46153b48         lbfgs      0.1            balanced             0.259          0.236511
5   d0044906bd04496cab882e4e977e00e3          saga      1.0            balanced             0.259          0.236511
6   bdef2c5cfa9d4b23ba5b0c5cbe2e57e7         lbfgs      0.1            balanced             0.259          0.236511
7   f9399c3458b242ab98daac7

## Step 3: Create Unified Prediction Pipeline

We combine the preprocessing pipeline (from Notebook 1) and the trained classification model (from Notebook 2) into a **single Scikit-learn Pipeline object**. This ensures:
- All preprocessing steps are applied consistently during inference
- The deployment artifact is self-contained and portable
- Raw input data can be fed directly without manual transformation

### Why Pipelines?
A Pipeline packages multiple processing steps into a single estimator. This is critical for production because:
1. It eliminates the risk of applying different preprocessing at training vs inference time
2. The Vertex AI pre-built container can load and serve a single Pipeline object
3. It makes the deployment reproducible and version-controlled

In [16]:
# Load the preprocessor and best classification model
preprocessor = joblib.load(os.path.join(ARTIFACTS_DIR, 'preprocessor.pkl'))
best_classifier = joblib.load(os.path.join(ARTIFACTS_DIR, 'best_segment_classifier.pkl'))
le_segment = joblib.load(os.path.join(ARTIFACTS_DIR, 'le_segment.pkl'))
split_data = joblib.load(os.path.join(ARTIFACTS_DIR, 'train_test_split.pkl'))

print('Preprocessor:', preprocessor)
print('\nClassifier:', best_classifier)
print('\nSegment classes:', le_segment.classes_)

Preprocessor: ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['quantity', 'order_month',
                                  'order_day_of_week', 'order_hour',
                                  'shipping_delay_days']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['category', 'payment_method', 'device_type',
                                  'channel'])])

Classifier: LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42,
                   solver='saga')

Segment classes: ['New' 'Returning' 'VIP']


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Feature order: category(0), quantity(1), payment_method(2), device_type(3),
#                channel(4), order_month(5), order_day_of_week(6), order_hour(7),
#                shipping_delay_days(8)
FEATURE_NAMES = ['category', 'quantity', 'payment_method', 'device_type',
                 'channel', 'order_month', 'order_day_of_week', 'order_hour',
                 'shipping_delay_days']
NUM_INDICES = [1, 5, 6, 7, 8]  # quantity, order_month, order_day_of_week, order_hour, shipping_delay_days
CAT_INDICES = [0, 2, 3, 4]     # category, payment_method, device_type, channel

# Rebuild preprocessor with INTEGER indices (works with numpy arrays)
deploy_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUM_INDICES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_INDICES)
    ]
)

# Fit the new preprocessor on training data (using the same column order)
X_train = split_data['X_train'][FEATURE_NAMES].values
X_test = split_data['X_test'][FEATURE_NAMES].values
y_train = split_data['y_segment_train_enc']

# Create the deployment pipeline
final_pipeline = Pipeline([
    ('preprocessor', deploy_preprocessor),
    ('classifier', best_classifier)
])

# Fit the pipeline (classifier is already trained, but preprocessor needs fitting)
deploy_preprocessor.fit(X_train)
# We only need to fit the preprocessor, the classifier is already trained
# So we set the classifier directly
final_pipeline.named_steps['classifier'] = best_classifier

# Verify with test data
X_test_transformed = deploy_preprocessor.transform(X_test)
y_pred = best_classifier.predict(X_test_transformed)

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(split_data['y_segment_test_enc'], y_pred)
print(f'\u2705 Deploy Pipeline Created (index-based, no custom functions)')
print(f'Accuracy on test set: {accuracy:.4f}')
print(final_pipeline)


✅ Deploy Pipeline Created (index-based, no custom functions)
Accuracy on test set: 0.2590
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  [1, 5, 6, 7, 8]),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [0, 2, 3, 4])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=42, solver='saga'))])


In [18]:
# Verify the full pipeline end-to-end with a raw array
test_array = ['Electronics', 2, 'Credit Card', 'Mobile', 'Organic', 6, 3, 14, 2.5]
y_pred = final_pipeline.predict([test_array])
y_pred_label = le_segment.inverse_transform(y_pred)
print(f'\u2705 Pipeline verification: {y_pred_label[0]}')

# Also verify with test set
X_test_arr = split_data['X_test'][FEATURE_NAMES].values
y_pred_all = final_pipeline.predict(X_test_arr)
accuracy = accuracy_score(split_data['y_segment_test_enc'], y_pred_all)
print(f'Test set accuracy: {accuracy:.4f}')


✅ Pipeline verification: Returning
Test set accuracy: 0.2590


## Step 4: Save the Model Artifact

We serialize the final pipeline using **joblib** as `model.joblib`. This format is required for compatibility with Vertex AI's pre-built Scikit-learn container images.

In [19]:
# Save the unified pipeline as model.joblib
model_path = os.path.join(ARTIFACTS_DIR, 'model.joblib')
joblib.dump(final_pipeline, model_path)

# Verify the saved model loads and predicts correctly
loaded_pipeline = joblib.load(model_path)
test_input = [['Electronics', 2, 'Credit Card', 'Mobile', 'Organic', 6, 3, 14, 2.5]]
y_pred_verify = loaded_pipeline.predict(test_input)

print(f'\u2705 Model saved as: {os.path.abspath(model_path)}')
print(f'   File size: {os.path.getsize(model_path) / 1024:.1f} KB')
print(f'   Verification: {le_segment.inverse_transform(y_pred_verify)}')

# Also save the label encoder for decoding predictions
joblib.dump(le_segment, os.path.join(ARTIFACTS_DIR, 'le_segment_deployment.pkl'))
print('\u2705 Label encoder saved for prediction decoding.')


✅ Model saved as: /home/kirito/self project/Apex_ML_Final_Project/artifacts/model.joblib
   File size: 4.2 KB
   Verification: ['Returning']
✅ Label encoder saved for prediction decoding.


In [20]:
# Verify requirements.txt exists
req_path = os.path.join(ARTIFACTS_DIR, 'requirements.txt')
print('=== requirements.txt ===')
with open(req_path, 'r') as f:
    print(f.read())

print('\n\u2705 Deployment artifacts ready:')
print(f'  1. {model_path}')
print(f'  2. {req_path}')

=== requirements.txt ===
scikit-learn==1.5.2
pandas==2.2.3
numpy>=2.0.0
joblib==1.4.2
imbalanced-learn==0.12.4


✅ Deployment artifacts ready:
  1. ../artifacts/model.joblib
  2. ../artifacts/requirements.txt


## Step 5: Upload Artifacts to Google Cloud Storage

The following code uploads `model.joblib` and `requirements.txt` to a Google Cloud Storage (GCS) bucket. This bucket will be referenced when importing the model into Vertex AI.

### Prerequisites
- Google Cloud SDK installed and authenticated (`gcloud auth login`)
- A GCS bucket created (e.g., `gs://auracart-ml-models/`)
- Billing enabled on the GCP project

> **Note:** The code cell below requires GCP credentials. Run it in your GCP-enabled environment.

In [21]:
# === Google Cloud Storage Upload ===
from google.cloud import storage

PROJECT_ID = 'apex-ml-cw'
BUCKET_NAME = 'apex-ml-models'
MODEL_DIR = 'customer_segment_model/v1'

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

blob_model = bucket.blob(f'{MODEL_DIR}/model.joblib')
blob_model.upload_from_filename(os.path.join(ARTIFACTS_DIR, 'model.joblib'))
print(f'Uploaded: gs://{BUCKET_NAME}/{MODEL_DIR}/model.joblib')

blob_req = bucket.blob(f'{MODEL_DIR}/requirements.txt')
blob_req.upload_from_filename(os.path.join(ARTIFACTS_DIR, 'requirements.txt'))
print(f'Uploaded: gs://{BUCKET_NAME}/{MODEL_DIR}/requirements.txt')

print('\n✅ All artifacts uploaded to GCS!')

Uploaded: gs://apex-ml-models/customer_segment_model/v1/model.joblib
Uploaded: gs://apex-ml-models/customer_segment_model/v1/requirements.txt

✅ All artifacts uploaded to GCS!


## Step 6: Deploy to Vertex AI

### Deployment Steps:

1. **Import Model** into Vertex AI Model Registry from GCS
2. **Select Container** — Scikit-learn pre-built prediction container
3. **Deploy to Endpoint** — Vertex AI manages the infrastructure
4. **Test the Endpoint** — Send a JSON prediction request

### Pre-built Container
Vertex AI provides a Scikit-learn pre-built container that:
- Automatically loads `model.joblib` from GCS
- Provides an HTTP server for serving predictions
- Handles request/response serialization

> **Note:** The code cells below require GCP credentials and Vertex AI API enabled.

In [22]:
from google.cloud import aiplatform

aiplatform.init(
    project=PROJECT_ID,
    location='asia-southeast1'
)

# MATCH scikit-learn version in requirements.txt (1.5.2)
SKLEARN_CONTAINER = 'asia-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest'

# Step 1: Upload the model
model = aiplatform.Model.upload(
    display_name='auracart-customer-segment-classifier',
    artifact_uri=f'gs://{BUCKET_NAME}/{MODEL_DIR}',
    serving_container_image_uri=SKLEARN_CONTAINER,
)
print(f'Model uploaded: {model.display_name}')

# Step 2: Deploy to an endpoint
endpoint = model.deploy(
    deployed_model_display_name='auracart-segment-v1',
    machine_type='n1-standard-2',
    min_replica_count=1,
    max_replica_count=1,
)
print(f'\n✅ Endpoint created: {endpoint.display_name}')


Creating Model
Create Model backing LRO: projects/935448168117/locations/asia-southeast1/models/618484086717022208/operations/3930748846618968064
Model created. Resource name: projects/935448168117/locations/asia-southeast1/models/618484086717022208@1
To use this Model in another session:
model = aiplatform.Model('projects/935448168117/locations/asia-southeast1/models/618484086717022208@1')
Model uploaded: auracart-customer-segment-classifier
Creating Endpoint
Create Endpoint backing LRO: projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632/operations/9217411859198509056
Endpoint created. Resource name: projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632')
Deploying model to Endpoint : projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632
Deploy Endpoint model backing

## Step 7: Test the Live Endpoint

We send a prediction request with a JSON payload representing a new e-commerce transaction. The endpoint should return the predicted customer segment.

In [25]:
# Test the deployed endpoint with a raw array
# Feature order: category, quantity, payment_method, device_type, channel,
#                order_month, order_day_of_week, order_hour, shipping_delay_days
test_instance = ['Electronics', 2, 'Credit Card', 'Mobile', 'Organic', 6, 3, 14, 2.5]

prediction = endpoint.predict(instances=[test_instance])
# The endpoint returns floats like [1.0] instead of ints [1]
# Just cast them to int before decoding
predicted_class = le_segment.inverse_transform([int(p) for p in prediction.predictions])
print('=== Prediction Request ===')
print(f'Input: {test_instance}')
print(f'\n=== Prediction Response ===')
print(f'Predicted Customer Segment: {predicted_class[0]}')


=== Prediction Request ===
Input: ['Electronics', 2, 'Credit Card', 'Mobile', 'Organic', 6, 3, 14, 2.5]

=== Prediction Response ===
Predicted Customer Segment: Returning


In [27]:
# Test with different customer profiles
test_samples = [
    ['Electronics', 2, 'Credit Card', 'Mobile', 'Organic', 6, 3, 14, 2.5],
    ['Clothing', 5, 'PayPal', 'Desktop', 'Email', 12, 5, 20, 5.0],
    ['Beauty', 1, 'Apple Pay', 'Tablet', 'Social', 1, 0, 8, 0.5],
]

prediction = endpoint.predict(instances=test_samples)
labels = le_segment.inverse_transform([int(p) for p in prediction.predictions])

for i, label in enumerate(labels):
    print(f'Transaction {i+1}: {label}')

Transaction 1: Returning
Transaction 2: New
Transaction 3: New


In [26]:
# Local simulation of the endpoint prediction
# This demonstrates the exact behavior the Vertex AI endpoint would produce

loaded_model = joblib.load(os.path.join(ARTIFACTS_DIR, 'model.joblib'))

# Create a sample transaction (matching the features expected by the pipeline)
sample_data = pd.DataFrame([{
    'category': 'Electronics',
    'quantity': 2,
    'payment_method': 'Credit Card',
    'device_type': 'Mobile',
    'channel': 'Organic',
    'order_month': 6,
    'order_day_of_week': 3,
    'order_hour': 14,
    'shipping_delay_days': 2.5
}])

print('=== Sample Transaction ===')
print(sample_data.to_string(index=False))

# Predict
prediction = loaded_model.predict(sample_data)
predicted_segment = le_segment.inverse_transform(prediction)

print(f'\n=== Prediction ===')
print(f'Predicted Customer Segment: {predicted_segment[0]}')

# Test with multiple samples
print('\n=== Batch Prediction Test ===')
test_samples = pd.DataFrame([
    {'category': 'Electronics', 'quantity': 1, 'payment_method': 'Credit Card',
     'device_type': 'Mobile', 'channel': 'Organic', 'order_month': 3,
     'order_day_of_week': 1, 'order_hour': 10, 'shipping_delay_days': 1.0},
    {'category': 'Clothing', 'quantity': 5, 'payment_method': 'PayPal',
     'device_type': 'Desktop', 'channel': 'Email', 'order_month': 12,
     'order_day_of_week': 5, 'order_hour': 20, 'shipping_delay_days': 5.0},
    {'category': 'Beauty', 'quantity': 3, 'payment_method': 'Apple Pay',
     'device_type': 'Tablet', 'channel': 'Social', 'order_month': 7,
     'order_day_of_week': 0, 'order_hour': 8, 'shipping_delay_days': 0.5},
])

batch_predictions = loaded_model.predict(test_samples)
batch_labels = le_segment.inverse_transform(batch_predictions)

for i in range(len(test_samples)):
    print(f'  Transaction {i+1}: {batch_labels[i]}')

print('\n\u2705 Pipeline accepts raw input and produces predictions successfully!')

=== Sample Transaction ===
   category  quantity payment_method device_type channel  order_month  order_day_of_week  order_hour  shipping_delay_days
Electronics         2    Credit Card      Mobile Organic            6                  3          14                  2.5

=== Prediction ===
Predicted Customer Segment: Returning

=== Batch Prediction Test ===
  Transaction 1: Returning
  Transaction 2: New
  Transaction 3: New

✅ Pipeline accepts raw input and produces predictions successfully!


In [28]:
# Undeploy and delete endpoint
endpoint.undeploy_all()
endpoint.delete()
print("✅ Endpoint deleted — no more charges!")

# Optional: delete model from registry too (it's free, but if you want clean)
# model.delete()

Undeploying Endpoint model: projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632
Undeploy Endpoint model backing LRO: projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632/operations/3448863686490324992
Endpoint model undeployed. Resource name: projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632
Deleting Endpoint : projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632
Endpoint deleted. . Resource name: projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632
Deleting Endpoint resource: projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632
Delete Endpoint backing LRO: projects/935448168117/locations/asia-southeast1/operations/5772158144260079616
Endpoint resource projects/935448168117/locations/asia-southeast1/endpoints/378322159908421632 deleted.
✅ Endpoint deleted — no more charges!


## Summary

In this notebook, we have:

1. **Reviewed MLflow experiments** and selected the champion model for customer segment prediction
2. **Created a unified Scikit-learn Pipeline** combining the preprocessing pipeline and trained classifier
3. **Verified the pipeline** works end-to-end with raw input data
4. **Serialized the model** as `model.joblib` using joblib
5. **Documented the GCS upload** and Vertex AI deployment process
6. **Tested the prediction pipeline** with sample e-commerce transactions

### Deployment Architecture
```
Raw Transaction Data
        ↓
  [Vertex AI Endpoint]
        ↓
  [model.joblib]
    ├── Preprocessor (ColumnTransformer)
    │   ├── StandardScaler (numerical features)
    │   └── OneHotEncoder (categorical features)
    └── LogisticRegression (multinomial/softmax)
        ↓
  Predicted Customer Segment
  (New / Returning / VIP)
```

### Next Steps for Production:
1. Uncomment and run the GCS upload cells with your GCP credentials
2. Deploy to Vertex AI and capture screenshots of the endpoint
3. Send test predictions and capture the response screenshots
4. Include all screenshots in the final technical report